In [1]:
# Loading Dataset from Huggingface & Imports
# 0. Imports
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 1. Loading Dataset
df = pd.read_csv("hf://datasets/maharshipandya/spotify-tracks-dataset/dataset.csv")
print(df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3           3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4           4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic          55       149610     False   
2              To Begin Again          57       210826     False   


In [2]:
# Dataset Description (Hady)
# 2. Size and Shape
print(f"Dataset Shape: {df.shape}")
print(f"Number of Records: {len(df)}")
print(f"Number of Features: {len(df.columns)}")

# 3. Feature Types and Names
print("\nFeature Information:")
print(df.info())

# 4. Class Labels and Distribution
print("\nClass Distribution:")
print(df['track_genre'].value_counts().sort_values(ascending=True))

# 5. Descriptive Statistics
print("\nDescriptive Statistics for Features:")
stats = df[['popularity', 'duration_ms', 'danceability', 'energy',
            'loudness', 'speechiness', 'acousticness', 'instrumentalness',
            'liveness', 'valence', 'tempo']].describe()
print(stats)
# ---

Dataset Shape: (114000, 21)
Number of Records: 114000
Number of Features: 21

Feature Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness     

In [3]:
# Preprocessing (Hady)
# 6. Check for Missing Values
print("\nMissing Values per Feature:")
print(df.isnull().sum())

# Finding and printing the rows with missing values
missing_row = df[df.isnull().any(axis=1)]
print("\nRecord/s with missing values:")
print(missing_row)

# Dropping missing record
df = df.dropna()

# Verifying that the record is gone
print(f"\nDataset shape: {df.shape}")
print(f"Total missing values remaining: {df.isnull().sum().sum()}")
# ---


Missing Values per Feature:
Unnamed: 0          0
track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

Record/s with missing values:
       Unnamed: 0                track_id artists album_name track_name  \
65900       65900  1kR4gIb7nGxHPI3D2ifs59     NaN        NaN        NaN   

       popularity  duration_ms  explicit  danceability  energy  ...  loudness  \
65900           0            0     False         0.501   0.583  ...     -9.46   

       mode  speechiness  acousticness  instrumentalness  liveness  valence  \
65900     0       0.0605          0.69           0.00396    0.0747    0.

In [4]:
# 7. Dropping metadata and csv id/index columns (Hady)
# List of columns to be removed
cols_to_drop = ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name']

# Dropping the metadata columns
df = df.drop(columns=cols_to_drop)

# Verifying the remaining features
print("Remaining columns in the dataset:")
print(df.columns.tolist())
print(f"\nShape after dropping columns: {df.shape}")
# ---

Remaining columns in the dataset:
['popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

Shape after dropping columns: (113999, 16)


In [5]:
# 8. Binary Encoding (Hady)
# Convert 'explicit' to binary (0 and 1)
df['explicit'] = df['explicit'].astype(int)
# df['mode'] = df['mode'].astype(int) - does not require conversion as is already in binary format

# Verifying the conversion
print("Values after conversion:")
print(df['explicit'].value_counts())
print(df['mode'].value_counts())
# ---

Values after conversion:
explicit
0    104252
1      9747
Name: count, dtype: int64
mode
1    72681
0    41318
Name: count, dtype: int64


In [ ]:
# 9. Feature Groups for Modeling
# Keep preprocessing logic declarative here.
# The actual encoding/scaling will happen inside a scikit-learn Pipeline after the train-test split.
continuous_features = [
    'popularity', 'duration_ms', 'danceability', 'energy',
    'loudness', 'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo'
]

binary_features = ['explicit', 'mode']
categorical_features = ['key', 'time_signature']
target_column = 'track_genre'

print('Continuous features:', continuous_features)
print('Binary features:', binary_features)
print('Categorical features:', categorical_features)
print('Target column:', target_column)
print(f'Total modeling features: {len(continuous_features) + len(binary_features) + len(categorical_features)}')
# ---


In [ ]:
# 10. Train-Test Split (clean genre classification setup)
from sklearn.model_selection import train_test_split

X = df[continuous_features + binary_features + categorical_features].copy()
y = df[target_column].copy()

# Downcast numeric dtypes before modeling to keep Colab RAM usage modest.
X[continuous_features] = X[continuous_features].astype(np.float32)
X[binary_features + categorical_features] = X[binary_features + categorical_features].astype(np.int16)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')
print(f'Number of genres: {y.nunique()}')
print(f'Feature table memory usage: {X.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
# ---


In [ ]:
# 11. Train-only preprocessing and Colab-friendly settings
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, f1_score

LOW_RAM_MODE = True
TUNING_SAMPLE_SIZE = 25000 if LOW_RAM_MODE else len(X_train)
SEARCH_ITERATIONS = 8 if LOW_RAM_MODE else 12
CV_FOLDS = 3

try:
    categorical_encoder = OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False,
        dtype=np.float32
    )
except TypeError:
    categorical_encoder = OneHotEncoder(
        handle_unknown='ignore',
        sparse=False,
        dtype=np.float32
    )

preprocessor = ColumnTransformer(
    transformers=[
        ('continuous', StandardScaler(), continuous_features),
        ('binary', 'passthrough', binary_features),
        ('categorical', categorical_encoder, categorical_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

def summarize_predictions(model_name, y_true, y_pred):
    return {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro')
    }

def top_confusion_pairs(y_true, y_pred, top_n=15):
    error_df = pd.DataFrame({'actual': y_true, 'predicted': y_pred})
    error_df = error_df[error_df['actual'] != error_df['predicted']]
    if error_df.empty:
        return pd.DataFrame(columns=['actual', 'predicted', 'count'])
    return (
        error_df
        .groupby(['actual', 'predicted'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(top_n)
    )

print('LOW_RAM_MODE:', LOW_RAM_MODE)
print('TUNING_SAMPLE_SIZE:', min(TUNING_SAMPLE_SIZE, len(X_train)))
print('SEARCH_ITERATIONS:', SEARCH_ITERATIONS)
print('CV_FOLDS:', CV_FOLDS)
print('Preprocessing is now fit on X_train only through the Pipeline.')
# ---


In [ ]:
# 12. Baseline Models (lighter and cleaner than the previous workflow)
import gc
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    'Gaussian Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(
        n_estimators=150,
        max_depth=None,
        min_samples_leaf=2,
        max_samples=0.8,
        random_state=42,
        n_jobs=1
    ),
    'Histogram Gradient Boosting': HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=200,
        max_leaf_nodes=31,
        min_samples_leaf=30,
        random_state=42
    )
}

results = []
best_baseline_name = None
best_baseline_f1 = -1.0

for name, estimator in models.items():
    model = Pipeline([
        ('preprocess', preprocessor),
        ('model', estimator)
    ])

    print()
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    summary = summarize_predictions(name, y_test, y_pred)
    results.append(summary)

    print(f"{name} Accuracy: {summary['accuracy']:.4f}")
    print(f"{name} Macro F1-score: {summary['macro_f1']:.4f}")

    if summary['macro_f1'] > best_baseline_f1:
        best_baseline_f1 = summary['macro_f1']
        best_baseline_name = name

    del model, y_pred
    gc.collect()

results_df = pd.DataFrame(results).sort_values(['macro_f1', 'accuracy'], ascending=False).reset_index(drop=True)
print()
print('Baseline model results:')
print(results_df.to_string(index=False))
print()
print(f'Best baseline model: {best_baseline_name}')
# ---



In [ ]:
# 13. RAM-friendly Random Forest tuning + final evaluation
import gc
from sklearn.model_selection import RandomizedSearchCV

# Tune only the Random Forest and do it on a stratified subset to keep Colab stable.
if len(X_train) > TUNING_SAMPLE_SIZE:
    X_tune, _, y_tune, _ = train_test_split(
        X_train,
        y_train,
        train_size=TUNING_SAMPLE_SIZE,
        stratify=y_train,
        random_state=42
    )
else:
    X_tune, y_tune = X_train, y_train

rf_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(random_state=42, n_jobs=1))
])

param_distributions = {
    'model__n_estimators': [120, 180, 260],
    'model__max_depth': [None, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 0.5],
    'model__max_samples': [0.7, 0.85, None]
}

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=SEARCH_ITERATIONS,
    scoring='f1_macro',
    cv=CV_FOLDS,
    n_jobs=1,
    random_state=42,
    verbose=1,
    refit=True
)

random_search.fit(X_tune, y_tune)
print('Best Random Forest parameters:', random_search.best_params_)
print('Best cross-validated Macro F1:', round(random_search.best_score_, 4))

# Refit the best configuration on the full training split.
best_rf_params = {
    key.replace('model__', ''): value
    for key, value in random_search.best_params_.items()
}

del random_search, rf_pipeline, X_tune, y_tune
gc.collect()

tuned_rf = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(
        random_state=42,
        n_jobs=1,
        **best_rf_params
    ))
])

tuned_rf.fit(X_train, y_train)
y_pred_best_rf = tuned_rf.predict(X_test)

final_accuracy = accuracy_score(y_test, y_pred_best_rf)
final_macro_f1 = f1_score(y_test, y_pred_best_rf, average='macro')
final_weighted_f1 = f1_score(y_test, y_pred_best_rf, average='weighted')

print()
print('Tuned Random Forest Test Accuracy:', round(final_accuracy, 4))
print('Tuned Random Forest Test Macro F1:', round(final_macro_f1, 4))
print('Tuned Random Forest Test Weighted F1:', round(final_weighted_f1, 4))

report_df = pd.DataFrame(
    classification_report(y_test, y_pred_best_rf, output_dict=True, zero_division=0)
).T

print()
print('Overall classification summary:')
print(report_df.loc[['macro avg', 'weighted avg'], ['precision', 'recall', 'f1-score', 'support']].to_string())

worst_genres = (
    report_df
    .drop(index=['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
    .sort_values('f1-score')
    .head(15)[['precision', 'recall', 'f1-score', 'support']]
)
print()
print('Lowest-performing genres on the test split:')
print(worst_genres.to_string())

print()
print('Top confusion pairs (cleaner than printing a 114x114 matrix):')
print(top_confusion_pairs(y_test, y_pred_best_rf).to_string(index=False))

# Inspect transformed feature importance from the tuned Random Forest.
feature_names = tuned_rf.named_steps['preprocess'].get_feature_names_out()
feature_importances = tuned_rf.named_steps['model'].feature_importances_
importance_df = (
    pd.DataFrame({'feature': feature_names, 'importance': feature_importances})
    .sort_values('importance', ascending=False)
    .head(15)
)

print()
print('Top transformed features by importance:')
print(importance_df.to_string(index=False))
# ---

